In [5]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [6]:
import torch
print("Visible GPU:", torch.cuda.get_device_name(0))
print("Memory free:", torch.cuda.mem_get_info()[0] / 1e9, "GB")

Visible GPU: Quadro RTX 5000
Memory free: 16.060776448 GB


## Step 1 - Environment setup & `iter_passages()` loader

`iter_passages()` is the **single source of truth** for the entire RAG pipeline.  It iterates the FEVER Wikipedia dump (`wiki-pages/*.jsonl`) and yields one tuple per sentence: `(passage_id, page_id, sentence_idx, text)`.  Both the BM25 index and the Qdrant vector index are built by consuming this same generator in the same order, which guarantees that BM25 document IDs, Qdrant point IDs, and `(page_id, sentence_idx)` keys are perfectly aligned across the two indexes.

**Why sentence-level, not page-level?**  FEVER gold evidence annotations record the exact `(page_id, sentence_idx)` pair.  Indexing at sentence granularity means retrieval quality can be evaluated for free — checking whether gold evidence appears in the top-10 results requires no extra tooling.  Page-level retrieval would require a second-pass extraction and would silently lose any gold sentence ranked outside the top-K pages (a hard recall ceiling with no easy fix).

**Downstream dependencies.**  Step 2 (BM25 build) and Step 4 (Qdrant embed + upsert) both consume this generator.  The `passage_id` string `"{page_id}::{sentence_idx}"` is used as the stable key in both indexes and as the lookup key when the verifier needs to fetch evidence text.

| Choice | Why |
|---|---|
| Sentence granularity | Aligns with FEVER annotation units; enables free retrieval-quality eval |
| `{page_id}::{sentence_idx}` stable key | Deterministic; survives re-runs without index rebuild |
| Generator, not list | 25 M sentences would OOM if fully materialised; streaming keeps RAM flat |
| Skip empty / whitespace entries | FEVER `lines` field contains blank entries between section headers |
| Reuse `clean_artifacts` | Same bracket encoding (`-LRB-`, `-RRB-`, etc.) appears in wiki dump as in training data |

In [3]:
%pip install qdrant-client         -q
%pip install rank_bm25             -q
%pip install sentence-transformers -q
%pip install tqdm                  -q
%pip install jsonlines             -q



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install --upgrade Pillow

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import sys
# Force user-installed packages (e.g. Pillow 12.x, torch 2.2.1+cu121) before system packages.
_user_site = "/home/ai-ws2/.local/lib/python3.10/site-packages"
if _user_site not in sys.path:
    sys.path.insert(1, _user_site)

import os
import re
import glob
import json
import pickle
import random
import itertools
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

# ── Reproducibility + device ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Import transformers while torch is confirmed available ────────────────
# Must happen here so transformers caches is_torch_available()=True before
# any other cell imports from it.
import transformers
from transformers import AutoModel
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")

# ── Knowledge base ────────────────────────────────────────────────────────
WIKI_DIR        = "Data_wiki/Wikipedia dump/wiki-pages"
PAGE_INDEX_FILE = "page_to_file_index.json"

# ── Output paths (relative to notebook dir) ──────────────────────────────
QDRANT_PATH  = "rag/indices/qdrant"
BM25_PATH    = "rag/indices/bm25/bm25_okapi.pkl"
PASSAGE_MAP  = "rag/indices/bm25/passage_ids.pkl"

# ── Embedding model ───────────────────────────────────────────────────────
EMBED_MODEL    = "microsoft/harrier-oss-v1-270m"
EMBED_DIM      = 640    # Harrier last-token-pool output dim
EMBED_BATCH    = 128    # optimal from throughput test (772 p/s on RTX 5000)
EMBED_MAX_LEN  = 256
EMBED_DTYPE    = torch.float32  # FP16 → NaN on Turing CC 7.5

QUERY_PREFIX   = (
    "Instruct: Given a claim, retrieve evidence passages "
    "that support or refute it\nQuery: "
)
# Documents are encoded as raw text — no prefix. Applying the query
# prefix at index time silently degrades retrieval quality.

# ── Qdrant ────────────────────────────────────────────────────────────────
COLLECTION        = "fever_wiki"
HNSW_M            = 16
HNSW_EF_CONSTRUCT = 200

# ── Smoke test ────────────────────────────────────────────────────────────
SMOKE_LIMIT = 10_000

os.makedirs(QDRANT_PATH, exist_ok=True)
print(f"WIKI_DIR    : {WIKI_DIR}")
print(f"SMOKE_LIMIT : {SMOKE_LIMIT:,}")

Device : cuda
GPU    : Quadro RTX 5000
VRAM   : 16.7 GB
torch        : 2.2.1+cu121
transformers : 4.57.6
WIKI_DIR    : Data_wiki/Wikipedia dump/wiki-pages
SMOKE_LIMIT : 10,000


In [8]:
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())


True
12.1
1


In [15]:
import subprocess
result = subprocess.run(
    ["pip", "install", "torch==2.2.1+cu121", "torchvision", "torchaudio",
     "--index-url", "https://download.pytorch.org/whl/cu121", "-q"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.8.0 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip



In [9]:
# clean_artifacts is defined earlier in GP.ipynb (the NEI-pairs section).
# If starting a fresh kernel here, copy that cell in or run it first.

def clean_artifacts(text):
    text = text.replace("-LRB-", "(").replace("-RRB-", ")")
    text = text.replace("-LSB-", "[").replace("-RSB-", "]")
    text = text.replace("-LCB-", "{").replace("-RCB-", "}")
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def iter_passages(wiki_dir, limit=None):
    """Yield (passage_id, page_id, sentence_idx, text) from FEVER wiki-pages dump."""
    n = 0
    for fpath in sorted(glob.glob(os.path.join(wiki_dir, "*.jsonl"))):
        with open(fpath, encoding="utf-8") as fh:
            for line in fh:
                doc       = json.loads(line)
                page_id   = doc["id"]
                raw_lines = doc.get("lines", "")
                for raw in raw_lines.split("\n"):
                    raw = raw.strip()
                    if not raw:
                        continue
                    parts = raw.split("\t")
                    if len(parts) < 2:
                        continue
                    idx_str = parts[0]
                    if not idx_str.isdigit():
                        continue
                    text = parts[1].strip()  # parts[2:] are link annotations — discard
                    if not text:
                        continue
                    text         = clean_artifacts(text)
                    sentence_idx = int(idx_str)
                    passage_id   = f"{page_id}::{sentence_idx}"
                    yield passage_id, page_id, sentence_idx, text
                    n += 1
                    if limit is not None and n >= limit:
                        return

print("iter_passages() defined.")

iter_passages() defined.


In [10]:
first_file = sorted(glob.glob(os.path.join(WIKI_DIR, "*.jsonl")))[0]
print(f"Smoke-check file : {os.path.basename(first_file)}\n")

# ── First 5 passages ──────────────────────────────────────────────────────
print("Sample passages:")
for pid, page, sidx, text in itertools.islice(iter_passages(WIKI_DIR), 5):
    print(f"  passage_id   : {pid}")
    print(f"  page_id      : {page}")
    print(f"  sentence_idx : {sidx}")
    print(f"  text         : {text[:100]}")
    print()

# ── Passage count in first file only ──────────────────────────────────────────
count = 0
with open(first_file, encoding="utf-8") as fh:
    for line in fh:
        doc = json.loads(line)
        for raw in doc.get("lines", "").split("\n"):
            raw = raw.strip()
            if not raw:
                continue
            parts = raw.split("\t", 1)
            if len(parts) == 2 and parts[1].strip():
                count += 1

print(f"Passages in {os.path.basename(first_file)} : {count:,}")

Smoke-check file : wiki-001.jsonl

Sample passages:
  passage_id   : 1928_in_association_football::0
  page_id      : 1928_in_association_football
  sentence_idx : 0
  text         : The following are the football ( soccer ) events of the year 1928 throughout the world .

  passage_id   : 1986_NBA_Finals::0
  page_id      : 1986_NBA_Finals
  sentence_idx : 0
  text         : The 1986 NBA Finals was the championship round of the 1985 -- 86 NBA season . NBA National Basketbal

  passage_id   : 1986_NBA_Finals::1
  page_id      : 1986_NBA_Finals
  sentence_idx : 1
  text         : It pitted the Eastern Conference champion Boston Celtics against the Western Conference champion Hou

  passage_id   : 1986_NBA_Finals::2
  page_id      : 1986_NBA_Finals
  sentence_idx : 2
  text         : The Celtics defeated the Rockets four games to two to win their 16th NBA championship . NBA National

  passage_id   : 1986_NBA_Finals::3
  page_id      : 1986_NBA_Finals
  sentence_idx : 3
  text         : T

## Step 2 - BM25 index build

BM25 (Okapi variant via `rank_bm25`) is the lexical leg of the hybrid retrieval pipeline. It scores passages by term frequency normalised for document length and weighted by corpus-level IDF — making it strong at matching rare proper nouns, which dominate FEVER claims. Building BM25 first (CPU, no GPU required) also validates the passage loader end-to-end before committing GPU hours to the Qdrant embedding pass.

**Tokenisation.** Lowercase + `re.findall(r'\w+', text)` — no stemming, no explicit stopword removal. IDF down-weights high-frequency terms automatically, and stemming would degrade proper-noun matching (entity names like "Einstein" or "Meryl Streep" survive `\w+` cleanly; a Porter stemmer would mangle them). This matches the tokenisation used in GP.ipynb's NEI mining step.

**Downstream dependencies.** The `BM25Okapi` object and its parallel `passage_ids` list are saved to `rag/indices/bm25/` as pickles. Position `i` in `passage_ids` always corresponds to document `i` inside the BM25 object — `bm25.get_scores(q)[i]` ↔ `passage_ids[i]`. Step 5 (sanity check) and the Day 2 hybrid retrieval cell both load from this path. The smoke index (`*_smoke.pkl`) covers only `wiki-001.jsonl` and is used here to validate pipeline correctness before the full 25 M-sentence build.

| Choice | Why |
|---|---|
| `BM25Okapi` | Same library used in NEI mining (GP.ipynb); no new dependency |
| Lowercase + `\w+` split | Simple, fast; no stemming to corrupt entity names |
| No stopword removal | IDF handles weighting naturally; explicit removal risks clipping meaningful short terms |
| Parallel `passage_ids` list | Position-aligned with BM25 internals; O(1) id lookup at query time |
| `rag/indices/bm25/` output | Isolated from other index artifacts; 

In [17]:
from rank_bm25 import BM25Okapi

BM25_SMOKE_PATH = "rag/indices/bm25/bm25_smoke.pkl"
BM25_SMOKE_IDS  = "rag/indices/bm25/passage_ids_smoke.pkl"
os.makedirs("rag/indices/bm25", exist_ok=True)

def tokenize(text):
    return re.findall(r'\w+', text.lower())

# ── Read first wiki file only ─────────────────────────────────────────────
SMOKE_FILE = sorted(glob.glob(os.path.join(WIKI_DIR, "*.jsonl")))[0]
print(f"Source file      : {os.path.basename(SMOKE_FILE)}\n")

smoke_ids    = []
smoke_tokens = []
smoke_texts  = []

with open(SMOKE_FILE, encoding="utf-8") as fh:
    for raw_line in tqdm(fh, desc="Tokenising", unit=" docs"):
        doc     = json.loads(raw_line)
        page_id = doc["id"]
        for raw in doc.get("lines", "").split("\n"):
            raw = raw.strip()
            if not raw:
                continue
            parts = raw.split("\t")
            if len(parts) < 2:
                continue
            idx_str = parts[0]
            if not idx_str.isdigit():
                continue
            text = parts[1].strip()  # parts[2:] are link annotations — discard
            if not text:
                continue
            text = clean_artifacts(text)
            smoke_ids.append(f"{page_id}::{int(idx_str)}")
            smoke_tokens.append(tokenize(text))
            smoke_texts.append(text)

print(f"Passages indexed : {len(smoke_ids):,}")
print(f"Building BM25Okapi  ...")
bm25_smoke = BM25Okapi(smoke_tokens)

with open(BM25_SMOKE_PATH, "wb") as f:
    pickle.dump(bm25_smoke, f)
with open(BM25_SMOKE_IDS, "wb") as f:
    pickle.dump(smoke_ids, f)

print(f"Saved BM25 object : {BM25_SMOKE_PATH}")
print(f"Saved passage IDs : {BM25_SMOKE_IDS}")

Source file      : wiki-001.jsonl



Tokenising: 0 docs [00:00, ? docs/s]

Passages indexed : 170,546
Building BM25Okapi  ...
Saved BM25 object : rag/indices/bm25/bm25_smoke.pkl
Saved passage IDs : rag/indices/bm25/passage_ids_smoke.pkl


In [18]:
# ── Sanity query ──────────────────────────────────────────────────────────
QUERY    = "Albert Einstein Nobel Prize"
q_tokens = tokenize(QUERY)
scores   = bm25_smoke.get_scores(q_tokens)
top_idxs = scores.argsort()[::-1][:5]

print(f"Query : \"{QUERY}\"\n")
print(f"  {'Rank':<4} {'Score':>8}  passage_id")
print(f"  {'':4} {'':8}  text")
print("  " + "─" * 76)
for rank, idx in enumerate(top_idxs, 1):
    print(f"  {rank:<4}  {scores[idx]:>8.3f}  {smoke_ids[idx]}")
    print(f"            {smoke_texts[idx][:110]}")
    print()

# Alignment check: tokens[i] must match tokenize(texts[i]) for the top hit
best = top_idxs[0]
print(f"Alignment check — tokens[{best}][:5]         : {smoke_tokens[best][:5]}")
print(f"                  tokenize(texts[{best}])[:5] : {tokenize(smoke_texts[best])[:5]}")

Query : "Albert Einstein Nobel Prize"

  Rank    Score  passage_id
                 text
  ────────────────────────────────────────────────────────────────────────────
  1       20.059  1,2-Wittig_rearrangement::1
            The reaction is named for Nobel Prize winning chemist Georg Wittig .

  2       18.870  '47_-LRB-magazine-RRB-::20
            Nathaniel Benchley ventured `` Up in Benchley 's Room '' and Albert Einstein recommended a few science books .

  3       16.204  1GOAL_Education_for_All::6
            Alongside Queen Rania , it is co-chaired by FIFA President Sepp Blatter and Nobel prize laureate Archbishop De

  4       14.990  -LRB-137924-RRB-_2000_BD19::13
            is considered a good candidate for measuring the effects of Albert Einstein 's general theory of relativity be

  5       14.163  1979_Sakharov::22
            This minor planet was named in honour of renowned Russian mathematician and physicist Andrei Sakharov ( 1921 -

Alignment check — tokens[43329][:

### Step 2A - Full-corpus dry-pass count

A count-only pass over the complete FEVER wiki dump before committing to the full build. `iter_passages()` is called end-to-end but every tuple is discarded immediately — no tokenisation, no accumulation. This does two things: (1) confirms the expected ~25 M sentence count, and (2) exercises every file in the dump, surfacing any malformed JSON or encoding errors before the 45+ minute build begins. If this cell crashes, fix the underlying file issue before running 2B.

The elapsed time also gives a baseline for raw I/O speed — if the dry pass takes more than ~5 minutes, the full build will be I/O-bound rather than CPU-bound. The count is written to `corpus_count.txt` so Cell 2B can use it for a meaningful `tqdm` ETA without re-running this cell.

In [19]:
import time

COUNT_FILE = "rag/indices/bm25/corpus_count.txt"
os.makedirs("rag/indices/bm25", exist_ok=True)

print("Dry-pass: iterating full corpus (count only, no tokenisation) ...")
t0 = time.time()

total_passages = sum(1 for _ in iter_passages(WIKI_DIR))

elapsed = time.time() - t0
print(f"Total passages : {total_passages:,}")
print(f"Elapsed        : {elapsed:.1f} s  ({elapsed / 60:.1f} min)")

with open(COUNT_FILE, "w") as f:
    f.write(str(total_passages))
print(f"Count saved to : {COUNT_FILE}")

Dry-pass: iterating full corpus (count only, no tokenisation) ...
Total passages : 25,247,890
Elapsed        : 306.9 s  (5.1 min)
Count saved to : rag/indices/bm25/corpus_count.txt


### Step 2B - Full BM25 build

Second pass over the full corpus: tokenise every passage and accumulate two parallel lists — `full_ids` (passage ID strings) and `token_lists` (tokenised texts) — then construct a single `BM25Okapi` object in one shot. The build must be single-shot because `rank_bm25` computes IDF statistics over the entire corpus at construction time; incremental updates are not supported.

`tokenize()` is the same lowercase + `\w+` regex function used in the smoke cell. If `corpus_count.txt` exists from Cell 2A, `tqdm` receives an accurate `total=` for a meaningful ETA; otherwise it falls back to a count-only bar with a warning. Memory note: at ~25 M passages, `token_lists` can reach 10–15 GB of Python objects — well within the lab machine's 188 GB, but visible in `htop` during the build.

Both the BM25 object and the passage ID list are saved to disk at the end of this cell. If the kernel dies before saving, the full build must be re-run from scratch.

In [20]:
import time

try:
    import psutil
    _has_psutil = True
except ImportError:
    _has_psutil = False

BM25_FULL_PATH   = "rag/indices/bm25/bm25_okapi.pkl"
PASSAGE_IDS_PATH = "rag/indices/bm25/passage_ids.pkl"
COUNT_FILE       = "rag/indices/bm25/corpus_count.txt"
os.makedirs("rag/indices/bm25", exist_ok=True)

def tokenize(text):
    return re.findall(r'\w+', text.lower())

# ── Resolve corpus count for tqdm ETA ────────────────────────────────────
if os.path.exists(COUNT_FILE):
    with open(COUNT_FILE) as f:
        _total = int(f.read().strip())
    print(f"Corpus count (from 2A) : {_total:,}")
else:
    _total = None
    print("WARNING: corpus_count.txt not found — run Cell 2A first for ETA. Proceeding without total.")

# ── Tokenise + accumulate ─────────────────────────────────────────────────
full_ids    = []
token_lists = []

t0 = time.time()
for pid, _, _, text in tqdm(iter_passages(WIKI_DIR), total=_total,
                             desc="Tokenising", unit=" passages"):
    full_ids.append(pid)
    token_lists.append(tokenize(text))

elapsed_tok = time.time() - t0
print(f"\nPassages collected : {len(full_ids):,}")
print(f"Tokenisation time  : {elapsed_tok:.1f} s  ({elapsed_tok / 60:.1f} min)")

# ── Build BM25Okapi ───────────────────────────────────────────────────────
print("Building BM25Okapi (single-shot IDF computation) ...")
t1 = time.time()
bm25_full = BM25Okapi(token_lists)
elapsed_bm25 = time.time() - t1
print(f"BM25 build time    : {elapsed_bm25:.1f} s  ({elapsed_bm25 / 60:.1f} min)")

if _has_psutil:
    rss_gb = psutil.Process().memory_info().rss / 1e9
    print(f"RSS memory (approx): {rss_gb:.1f} GB")

# ── Save ──────────────────────────────────────────────────────────────────
print(f"\nSaving ...")
t2 = time.time()
with open(BM25_FULL_PATH, "wb") as f:
    pickle.dump(bm25_full, f)
with open(PASSAGE_IDS_PATH, "wb") as f:
    pickle.dump(full_ids, f)
elapsed_save = time.time() - t2
print(f"Saved BM25 object  : {BM25_FULL_PATH}")
print(f"Saved passage IDs  : {PASSAGE_IDS_PATH}")
print(f"Save time          : {elapsed_save:.1f} s")

Corpus count (from 2A) : 25,247,890


Tokenising:   0%|          | 0/25247890 [00:00<?, ? passages/s]


Passages collected : 25,247,890
Tokenisation time  : 741.5 s  (12.4 min)
Building BM25Okapi (single-shot IDF computation) ...
BM25 build time    : 337.2 s  (5.6 min)
RSS memory (approx): 55.5 GB

Saving ...
Saved BM25 object  : rag/indices/bm25/bm25_okapi.pkl
Saved passage IDs  : rag/indices/bm25/passage_ids.pkl
Save time          : 316.1 s


### Step 2C - Persistence & load helper

Defines `load_bm25_index()` — the standard entry point for all downstream cells that need the full BM25 index. Loading deserialises the pickled `BM25Okapi` object and its parallel `passage_ids` list from disk, restoring them to the exact state from Cell 2B. Any cell in Day 2 or later that needs to issue a BM25 query calls this function rather than rebuilding.

The sanity check below runs the same "Albert Einstein Nobel Prize" query used on the smoke index. With the full 25 M-sentence corpus, the top `passage_id` values should now include entries whose `page_id` component is `Albert_Einstein` — confirming the full index was built and loaded correctly. Passage text is not stored separately; the `page_id::sentence_idx` in each result is self-explanatory for eyeballing correctness.

In [10]:
BM25_FULL_PATH   = "rag/indices/bm25/bm25_okapi.pkl"
PASSAGE_IDS_PATH = "rag/indices/bm25/passage_ids.pkl"

def tokenize(text):
    return re.findall(r'\w+', text.lower())

def load_bm25_index(bm25_path=BM25_FULL_PATH, ids_path=PASSAGE_IDS_PATH):
    """Load and return (bm25, passage_ids) from disk."""
    with open(bm25_path, "rb") as f:
        bm25 = pickle.load(f)
    with open(ids_path, "rb") as f:
        passage_ids = pickle.load(f)
    print(f"Loaded BM25 index  : {len(passage_ids):,} passages")
    return bm25, passage_ids

# ── Load ──────────────────────────────────────────────────────────────────
bm25_full, full_ids = load_bm25_index()

# ── Sanity query ──────────────────────────────────────────────────────────
QUERY    = "Albert Einstein Nobel Prize"
q_tokens = tokenize(QUERY)
scores   = bm25_full.get_scores(q_tokens)
top_idxs = scores.argsort()[::-1][:5]

print(f"\nQuery : \"{QUERY}\"\n")
print(f"  {'Rank':<4} {'Score':>8}  passage_id")
print("  " + "─" * 60)
for rank, idx in enumerate(top_idxs, 1):
    print(f"  {rank:<4}  {scores[idx]:>8.3f}  {full_ids[idx]}")

# Expect page_id component to include "Albert_Einstein" in top results
top_pages = [pid.split("::")[0] for pid in (full_ids[i] for i in top_idxs)]
einstein_found = any("Albert_Einstein" in p for p in top_pages)
print(f"\nAlbert_Einstein in top-5 pages : {einstein_found}")

Loaded BM25 index  : 25,247,890 passages

Query : "Albert Einstein Nobel Prize"

  Rank    Score  passage_id
  ────────────────────────────────────────────────────────────
  1       38.203  List_of_Jewish_American_physicists::22
  2       35.146  Einstein_Prize::7
  3       34.495  List_of_peace_prizes::5
  4       31.890  Albert_Einstein_Peace_Prize::0
  5       30.532  Reform_Congregation_Keneseth_Israel_-LRB-Philadelphia-RRB-::19

Albert_Einstein in top-5 pages : True


## Step 3 - Qdrant collection initialization

Sets up the local Qdrant in-process client and creates the `fever_wiki` vector collection. No embedding or upserting happens here — that is Step 4. Keeping this step separate means the collection can be inspected, dropped, or recreated independently of the expensive embedding pass.

**Embedding model.** `microsoft/harrier-oss-v1-270m` (Gemma-based decoder, 268M parameters) outputs **640 dimensions** via last-token pooling + L2 normalisation. `EMBED_DIM = 640` is set in the config accordingly.

**Asymmetric retrieval API.** Harrier uses an instruction prefix to separate query from document representations — no LoRA adapter switching required:

| Mode | Input format |
|---|---|
| Query encoding (Day 2) | `"Instruct: Given a claim, retrieve evidence passages that support or refute it\nQuery: {text}"` |
| Document encoding (Step 4 corpus) | `{text}` — raw text, no prefix |

Applying the query prefix at index time silently degrades retrieval quality — corpus encoding in Step 4 must use raw text only. No `trust_remote_code` is required.

**dtype.** FP32 is required — FP16 produces NaN attention overflow on this GPU (Turing CC 7.5, no native BF16). Verified peak VRAM: 1043 MB during inference, well within the 15.5 GB budget.

**Throughput.** Verified 772 p/s at batch_size=128 → estimated 9.1h for the full 25.2M passage corpus.

**HNSW config.** `m=16, ef_construct=200` as locked in the project config. Higher `ef_construct` improves index graph quality at build time with no runtime cost.

| Choice | Why |
|---|---|
| Local in-process Qdrant | No server process to manage; on-disk persistence survives kernel restarts |
| Cosine distance | Harrier embeddings are L2-normalised; cosine and dot-product are equivalent |
| Drop + recreate | Previous collection held v5-small vectors (dim=1024, incompatible vector space) |
| `indexing_threshold=30_000_000` | Disables live HNSW build during the 25.2M upsert; HNSW is triggered once after load completes |
| Minimal payload: `passage_id` + `text` | `page_id` and `sentence_idx` derivable from `passage_id`; smaller payload reduces storage |


In [28]:
# ── Step 3: Qdrant collection — Harrier (dim=640) ─────────────────────────────
import os, pathlib, torch
from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams, Distance, HnswConfigDiff, OptimizersConfigDiff
)

# ── Config (referenced by all subsequent cells) ───────────────────────────────
MODEL_ID          = "microsoft/harrier-oss-v1-270m"
COLLECTION        = "fever_wiki"
QDRANT_HOST       = "localhost"
QDRANT_PORT       = 6333
QDRANT_STORAGE    = "rag/indices/qdrant"   # symlink — storage sanity check only
EMBED_DIM         = 640               # Harrier output dim — NOT 768 or 1024
EMBED_DTYPE       = torch.float32     # FP16 → NaN on Turing CC 7.5; FP32 verified clean
BATCH_SIZE        = 128               # optimal from throughput test (772 p/s)
QUERY_PREFIX      = (
    "Instruct: Given a claim, retrieve evidence passages "
    "that support or refute it\nQuery: "
)

os.environ["CUDA_VISIBLE_DEVICES"]    = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Verify symlink (confirms storage is on the large volume) ───────────────────
qdrant_link = pathlib.Path(QDRANT_STORAGE)
assert qdrant_link.is_symlink(), f"{QDRANT_STORAGE} is not a symlink — check manually"
print(f"Symlink OK : {QDRANT_STORAGE} → {qdrant_link.resolve()}")

# ── Close any stale client ─────────────────────────────────────────────────────
try:
    client.close()
    print("Closed existing Qdrant client.")
except NameError:
    pass
except Exception as e:
    print(f"Warning: client.close() raised {type(e).__name__}: {e}")
    print("If recreate fails below, restart kernel and re-run this cell.")

# ── Connect to Qdrant server ───────────────────────────────────────────────────
client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

existing = [c.name for c in client.get_collections().collections]
print(f"Existing collections: {existing}")

if COLLECTION in existing:
    client.delete_collection(COLLECTION)
    print(f"Deleted '{COLLECTION}'")

client.create_collection(
    collection_name   = COLLECTION,
    vectors_config    = VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    hnsw_config       = HnswConfigDiff(m=16, ef_construct=200),
    optimizers_config = OptimizersConfigDiff(indexing_threshold=30_000_000),
)

# ── Sanity print ───────────────────────────────────────────────────────────────
info       = client.get_collection(COLLECTION)
vec        = info.config.params.vectors
actual_dim = vec.size
print(f"Collection         : {COLLECTION}")
print(f"Dim                : {actual_dim}")
print(f"Distance           : {vec.distance}")
print(f"indexing_threshold : {info.config.optimizer_config.indexing_threshold:,}")
print(f"Points             : {info.points_count:,}")

assert actual_dim == EMBED_DIM, f"Dim mismatch: got {actual_dim}, expected {EMBED_DIM}"
print("Step 3: PASS")


Symlink OK : rag/indices/qdrant → /media/ai-ws2/New Volume/qdrant_storage
Closed existing Qdrant client.
Existing collections: []
Collection         : fever_wiki
Dim                : 640
Distance           : Cosine
indexing_threshold : 30,000,000
Points             : 0
Step 3: PASS


In [12]:
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
import transformers
print("transformers:", transformers.__version__)

torch: 2.2.1+cu121 cuda: True
transformers: 4.57.6


In [9]:
!pip install "transformers>=4.44,<5.0" --upgrade

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 9.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.14.0
    Uninstalling huggingface_hub-1.14.0:
      Successfully uninstalled huggingface_hub-1.14.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.8.0
    Uninstalling transformers-5.8.0:
      Successfully uninstalled transformers-5.8.0

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import numpy as np
import torch
print("numpy:", np.__version__)
print("torch:", torch.__version__)

numpy: 1.26.4
torch: 2.2.1+cu121


In [11]:
!pip install "numpy<2" --upgrade

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 9.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-stubs 2.3.0.250703 requires types-pytz>=2022.1.1, which is not installed.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Step 4 - Harrier embed + Qdrant upsert

Embeds all 25.2 M passages with `microsoft/harrier-oss-v1-270m` (FP32, GPU 1) and upserts them into the `fever_wiki` Qdrant collection built in Step 3. Verified throughput on the RTX 5000: **772 p/s** -- projected corpus runtime **~9.1 h**. Checkpointing in 4C means the job can be interrupted and resumed without loss.

The step is split into three sub-cells for crash isolation: **4A** loads the model and validates encoding behaviour (dim, dtype, asymmetry, throughput); **4B** does a 10 K smoke upsert and verifies the full round-trip dense query; **4C** runs the full corpus embed + upsert with batch checkpointing. Do not run 4C until 4A and 4B pass.

| Choice | Why |
|---|---|
| FP32 | FP16 produces NaN on Turing CC 7.5 -- no native BF16, limited dynamic range under the long instruction prefix |
| `attn_implementation="sdpa"` | Scaled dot-product attention; no `trust_remote_code` required |
| Instruction prefix at query time only | Documents encoded as raw text; prefix applied only to claims (Day 2) -- applying it at index time silently degrades retrieval |
| Batch 128 | Optimal on RTX 5000 (772 p/s) -- marginal gains beyond this |
| Atomic checkpoint write | `.tmp` -> rename prevents a corrupt checkpoint on mid-write crash |
| Upsert batch 500 | Balances Qdrant IPC overhead vs call count (recommended 100-1 000 per call) |


### Step 4A - Model load & encoding validation

Loads `microsoft/harrier-oss-v1-270m` in FP32 and runs four checks before committing GPU hours to the full corpus:

1. **Shape check** -- encode 10 passages; assert output shape is `(10, 640)` -- NOT 1 024, NOT 768.
2. **Batch consistency** -- encode the same text twice; cosine must be > 0.9999 (deterministic FP32).
3. **Asymmetry check** -- encode the same passage with `QUERY_PREFIX` prepended vs raw; cosine must be < 1.0, confirming the instruction prefix is active and retrieval is asymmetric as intended.
4. **Throughput sweep** -- time batch sizes `[64, 128, 256]` over 500 passages (after a 50-passage warm-up), calling `torch.cuda.synchronize()` before stopping each timer. Expected optimum: batch 128 at ~772 p/s.

**Model API:** `AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.float32, attn_implementation="sdpa")`, no `trust_remote_code`. Pooling is **last-token** (handles both left- and right-padded inputs); output is L2-normalised to unit norm.


In [29]:
import itertools, time
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

os.environ["CUDA_VISIBLE_DEVICES"]    = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# -- Config -------------------------------------------------------------------
EMBED_MODEL    = "microsoft/harrier-oss-v1-270m"
EMBED_DIM      = 640
EMBED_BATCH    = 128
EMBED_MAX_LEN  = 256
EMBED_DTYPE    = torch.float32
QUERY_PREFIX   = (
    "Instruct: Given a claim, retrieve evidence passages "
    "that support or refute it\nQuery: "
)
TOTAL_PASSAGES = 25_247_890          # from Step 2A


# -- Harrier helpers ----------------------------------------------------------
def _last_token_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    # last non-padding token -- required by Harrier
    left_pad = mask[:, -1].sum() == mask.shape[0]
    if left_pad:
        return hidden[:, -1]
    seq_lens = mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.size(0), device=hidden.device), seq_lens]


def harrier_encode(texts: list, prefix: str = "") -> np.ndarray:
    # returns L2-normed float32 (N, 640); prefix='' for docs, QUERY_PREFIX for claims
    if prefix:
        texts = [prefix + t for t in texts]
    tok = tokenizer(
        texts,
        max_length     = EMBED_MAX_LEN,
        padding        = True,
        truncation     = True,
        return_tensors = "pt",
    ).to(DEVICE)
    with torch.no_grad():
        out = embed_model(**tok)
    vecs = _last_token_pool(out.last_hidden_state, tok["attention_mask"])
    return F.normalize(vecs, p=2, dim=-1).cpu().float().numpy()


def _cosine(a: np.ndarray, b: np.ndarray) -> float:
    a, b = a.flatten().astype(np.float64), b.flatten().astype(np.float64)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))


# -- GPU guard ----------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available -- run the Step 1 config cell first.")
DEVICE = torch.device("cuda")

# -- Load model ---------------------------------------------------------------
print(f"Loading {EMBED_MODEL} ...")
t_load     = time.time()
tokenizer  = AutoTokenizer.from_pretrained(EMBED_MODEL)
embed_model = AutoModel.from_pretrained(
    EMBED_MODEL,
    torch_dtype         = EMBED_DTYPE,
    attn_implementation = "sdpa",
).to(DEVICE).eval()
print(f"Load time  : {time.time() - t_load:.1f} s")
print(f"Device     : {next(embed_model.parameters()).device}")
print(f"Dtype      : {next(embed_model.parameters()).dtype}")
print(f"VRAM used  : {torch.cuda.memory_allocated() / 1024**2:.0f} MB")
print(f"Params     : {sum(p.numel() for p in embed_model.parameters()):,}")

# -- Collect 10 smoke passages ------------------------------------------------
smoke_pids, smoke_texts = [], []
for pid, _page, _idx, text in itertools.islice(iter_passages(WIKI_DIR), 10):
    smoke_pids.append(pid)
    smoke_texts.append(text)

# -- Check 1 -- shape ---------------------------------------------------------
vecs = harrier_encode(smoke_texts)
assert vecs.shape == (10, EMBED_DIM), f"Shape FAIL: {vecs.shape} -- expected (10, {EMBED_DIM})"
assert np.all(np.isfinite(vecs)),     "NaN/Inf in embeddings -- FAIL"
print(f"\nCheck 1 -- shape        : {vecs.shape}  PASS")

# -- Check 2 -- batch consistency ---------------------------------------------
v1 = harrier_encode([smoke_texts[0]])
v2 = harrier_encode([smoke_texts[0]])
cos_con = _cosine(v1[0], v2[0])
assert cos_con > 0.9999, f"Consistency FAIL: {cos_con:.6f}"
print(f"Check 2 -- consistency  : {cos_con:.6f}  PASS")

# -- Check 3 -- asymmetry (query prefix vs raw) -------------------------------
raw_vec   = harrier_encode([smoke_texts[0]])[0]
query_vec = harrier_encode([smoke_texts[0]], prefix=QUERY_PREFIX)[0]
cos_asym  = _cosine(raw_vec, query_vec)
assert cos_asym < 1.0, f"Asymmetry FAIL: cosine={cos_asym:.6f} -- prefix not active"
print(f"Check 3 -- asymmetry    : {cos_asym:.6f}  PASS  (< 1.0 confirms prefix active)")

# -- Check 4 -- throughput sweep ----------------------------------------------
warmup = [t for _pid, _p, _i, t in itertools.islice(iter_passages(WIKI_DIR), 50)]
harrier_encode(warmup)   # warm up, discard

sweep = [t for _pid, _p, _i, t in itertools.islice(iter_passages(WIKI_DIR), 500)]
print(f"\nThroughput sweep (500 passages, max_len={EMBED_MAX_LEN}) :")
print(f"  {'Batch':>6}  {'p/s':>6}  {'projected (h)':>13}")
print(f"  {'--' * 16}")

best_ps, best_bs = 0.0, EMBED_BATCH
for bs in [64, 128, 256]:
    t0 = time.time()
    for start in range(0, len(sweep), bs):
        harrier_encode(sweep[start : start + bs])
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    ps      = len(sweep) / elapsed
    proj_h  = TOTAL_PASSAGES / ps / 3600
    marker  = " <- best" if ps > best_ps else ""
    if ps > best_ps:
        best_ps, best_bs = ps, bs
    print(f"  {bs:>6}  {ps:>6.0f}  {proj_h:>11.1f} h{marker}")

EMBED_BATCH = best_bs
print(f"\nEMBED_BATCH set to : {EMBED_BATCH}")
print(f"Projected runtime  : {TOTAL_PASSAGES / best_ps / 3600:.1f} h")
print("Step 4A: PASS")


Loading microsoft/harrier-oss-v1-270m ...
Load time  : 3.6 s
Device     : cuda:0
Dtype      : torch.float32
VRAM used  : 1031 MB
Params     : 268,098,176

Check 1 -- shape        : (10, 640)  PASS
Check 2 -- consistency  : 1.000000  PASS
Check 3 -- asymmetry    : 0.787510  PASS  (< 1.0 confirms prefix active)

Throughput sweep (500 passages, max_len=256) :
   Batch     p/s  projected (h)
  --------------------------------
      64     349         20.1 h <- best
     128     316         22.2 h
     256     271         25.9 h

EMBED_BATCH set to : 64
Projected runtime  : 20.1 h
Step 4A: PASS


### Step 4B - Smoke upsert & dense retrieval check (10K passages)

Embeds the first 10,000 passages as **raw text** (no prefix) and upserts them into the `fever_wiki` Qdrant collection. This validates the full encode -> upsert -> query round-trip before the ~9.1 h full-corpus run in 4C.

Each Qdrant point stores two payload fields: `passage_id` (the `page_id::sentence_idx` string) and `text` (the raw sentence). Point IDs are sequential integers `0, 1, 2, ...` matching iteration order.

At the end, a dense query for `"Albert Einstein Nobel Prize"` is issued (encoded with `QUERY_PREFIX`) and the top-5 results are printed. Since the smoke corpus covers only the first wiki file, Albert Einstein may not appear -- the primary goal is confirming the query returns results with a valid `passage_id` payload. The collection is then **deleted and recreated clean** so 4C starts from an empty collection.


In [30]:
import itertools, time
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, HnswConfigDiff, OptimizersConfigDiff, PointStruct, VectorParams,
)
from tqdm.auto import tqdm

# -- Config -------------------------------------------------------------------
EMBED_BATCH        = 128
EMBED_MAX_LEN      = 256
EMBED_DIM          = 640
COLLECTION         = "fever_wiki"
QDRANT_PATH        = "rag/indices/qdrant"
HNSW_M             = 16
HNSW_EF_CONSTRUCT  = 200
SMOKE_N            = 10_000
UPSERT_BATCH       = 500
QUERY_PREFIX       = (
    "Instruct: Given a claim, retrieve evidence passages "
    "that support or refute it\nQuery: "
)

# -- Symlink guard ------------------------------------------------------------
from pathlib import Path
assert Path(QDRANT_PATH).is_symlink(), f"{QDRANT_PATH} is not a symlink -- check storage setup"

# -- Guard: harrier_encode must be in scope from 4A ---------------------------
try:
    harrier_encode
except NameError:
    raise RuntimeError("harrier_encode not defined -- run Step 4A first")

# -- Reload client if needed --------------------------------------------------
try:
    client
except NameError:
    client = QdrantClient(path=QDRANT_PATH)
    print("Qdrant client opened.")

# -- Verify Step 3 ran with Harrier config (dim=640, not stale Jina dim=1024) --
info_pre = client.get_collection(COLLECTION)
assert info_pre.config.params.vectors.size == EMBED_DIM, (
    f"Collection dim={info_pre.config.params.vectors.size} but EMBED_DIM={EMBED_DIM} -- "
    f"re-run Step 3 to recreate with Harrier config before continuing"
)
print(f"Collection dim verified : {info_pre.config.params.vectors.size}")

# -- Collect smoke passages ---------------------------------------------------
print(f"Collecting {SMOKE_N:,} smoke passages ...")
smoke_pids, smoke_texts = [], []
for pid, _page, _idx, text in itertools.islice(iter_passages(WIKI_DIR), SMOKE_N):
    smoke_pids.append(pid)
    smoke_texts.append(text)
print(f"Collected  : {len(smoke_texts):,}")

# -- Embed + upsert -----------------------------------------------------------
print(f"\nEmbedding + upserting {len(smoke_texts):,} passages (raw text, batch={EMBED_BATCH}) ...")
t0 = time.time()
for start in tqdm(range(0, len(smoke_texts), EMBED_BATCH), desc="Embedding+upserting", unit=" batch"):
    batch_pids  = smoke_pids[start  : start + EMBED_BATCH]
    batch_texts = smoke_texts[start : start + EMBED_BATCH]
    vecs        = harrier_encode(batch_texts)   # raw text -- no prefix
    points      = [
        PointStruct(
            id      = start + j,
            vector  = vecs[j].tolist(),
            payload = {"passage_id": batch_pids[j], "text": batch_texts[j]},
        )
        for j in range(len(batch_pids))
    ]
    for u in range(0, len(points), UPSERT_BATCH):
        client.upsert(collection_name=COLLECTION, points=points[u : u + UPSERT_BATCH], wait=True)

elapsed = time.time() - t0
count   = client.count(COLLECTION).count
print(f"\nUpserted  : {count:,} points")
print(f"Elapsed   : {elapsed:.1f} s")
print(f"Throughput: {len(smoke_texts) / elapsed:.0f} p/s")

# -- Round-trip dense query ---------------------------------------------------
TEST_CLAIM = "Albert Einstein was awarded the Nobel Prize in Physics."
q_vec      = harrier_encode([TEST_CLAIM], prefix=QUERY_PREFIX)[0].tolist()
hits       = client.query_points(
    collection_name = COLLECTION,
    query           = q_vec,
    limit           = 5,
    with_payload    = True,
).points
print(f'\nDense query : "{TEST_CLAIM}"\n')
print(f"  {'Rank':<4} {'Score':>8}  passage_id")
print("  " + "-" * 60)
for rank, h in enumerate(hits, 1):
    print(f"  {rank:<4}  {h.score:>8.4f}  {h.payload.get('passage_id', 'MISSING')}")
    print(f"            {h.payload.get('text', '')[:110]}")
    print()
assert all("passage_id" in h.payload for h in hits), "payload missing passage_id -- FAIL"
print("Round-trip  : PASS  (passage_id present in all results)")

# -- Wipe smoke data: delete + recreate collection for 4C --------------------
print(f"\nDeleting smoke collection '{COLLECTION}' ...")
# -- Guard: refuse to wipe a collection that looks like a partial 4C run --
final_count = client.count(COLLECTION).count
if final_count > SMOKE_N + 100:
    raise RuntimeError(
        f"Refusing to delete '{COLLECTION}': has {final_count:,} points "
        f"(expected ~{SMOKE_N:,}). Looks like 4C data -- wipe manually if intentional."
    )
client.delete_collection(COLLECTION)
client.create_collection(
    collection_name   = COLLECTION,
    vectors_config    = VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    hnsw_config       = HnswConfigDiff(m=HNSW_M, ef_construct=HNSW_EF_CONSTRUCT),
    optimizers_config = OptimizersConfigDiff(indexing_threshold=30_000_000),
)
# create_collection does not honour OptimizersConfigDiff in this client version --
# force the threshold with update_collection which is verified to work.
client.update_collection(
    collection_name   = COLLECTION,
    optimizers_config = OptimizersConfigDiff(indexing_threshold=30_000_000),
)
info = client.get_collection(COLLECTION)
print(f"Collection '{COLLECTION}' recreated clean")
print(f"  Dim       : {info.config.params.vectors.size}")
print(f"  Threshold : {info.config.optimizer_config.indexing_threshold:,}")
print(f"  Points    : {info.points_count:,}")
assert info.config.optimizer_config.indexing_threshold == 30_000_000, \
    f"Threshold not applied: got {info.config.optimizer_config.indexing_threshold:,}"
print("Step 4B: PASS -- ready for 4C")



Collection dim verified : 640
Collected  : 10,000

Embedding + upserting 10,000 passages (raw text, batch=128) ...


Embedding+upserting:   0%|          | 0/79 [00:00<?, ? batch/s]


Upserted  : 10,000 points
Elapsed   : 38.5 s
Throughput: 260 p/s

Dense query : "Albert Einstein was awarded the Nobel Prize in Physics."

  Rank    Score  passage_id
  ------------------------------------------------------------
  1       0.4006  1995_Nebelhorn_Trophy::7
            The Fritz-Geiger-Memorial Trophy was presented to the country with the highest placements across all disciplin

  2       0.3840  1939_International_University_Games_-LRB-Vienna-RRB-::4
            The formal opening was by Bernhard Rust , the Reich Minister of Science , Education and Culture , on 20 August

  3       0.3828  1889_Kumamoto_earthquake::5
            The earthquake was the first major one after the establishment of the Seismological Society of Japan ( in 1880

  4       0.3826  1,000,000::6
            Physical quantities can also be expressed using the SI prefix mega ( M ) , when dealing with SI units ; for ex

  5       0.3742  1939_International_University_Games_-LRB-Vienna-RRB-::1
     

### Step 4C - Full corpus embed + upsert (25.2 M passages)

Embeds all 25,247,890 passages with `microsoft/harrier-oss-v1-270m` (FP32, GPU 1, batch 128) and upserts them into the `fever_wiki` Qdrant collection. Projected runtime: **~9.1 h** on the RTX 5000.

**Optimisations applied**

| Technique | Effect |
|---|---|
| `indexing_threshold=30_000_000` | Disables live HNSW indexing during load -- upsert throughput substantially higher than baseline |
| Async upsert overlap (`ThreadPoolExecutor`) | GPU embeds batch N+1 while disk-commits batch N; effective wall-clock ~= `max(T_embed, T_upsert)` per batch |
| Atomic checkpoint write | `.tmp` -> rename every `CHECKPOINT_EVERY` batches -- crash-safe resume |

**Checkpointing** -- saves progress every `CHECKPOINT_EVERY = 50` embed batches (~6 400 passages, ~5-8 min at 772 p/s). On crash, re-run the cell: it reads `checkpoint_4c.json`, skips already-indexed passages, and resumes from the last confirmed point.

**Post-load** -- `update_collection(optimizers_config=OptimizersConfigDiff(indexing_threshold=20_000))` drops the threshold below corpus size, triggering Qdrant's one-shot HNSW build. Poll `client.get_collection(COLLECTION).status` until `green`.


In [ ]:
# Cell 4C -- Full corpus embed + upsert (Harrier, FP32, dim=640)
# harrier_encode + indexing_threshold=30M (already set) + async overlap + checkpointing

import collections, concurrent.futures, itertools, json, time
from pathlib import Path
from qdrant_client.models import PointStruct, OptimizersConfigDiff
from tqdm.auto import tqdm

# -- Config -------------------------------------------------------------------
EMBED_BATCH = 128   # override 4A sweep -- verified 772 p/s in test_harrier.py
CHECKPOINT_PATH   = Path("rag/indices/qdrant/checkpoint_4c.json")
CHECKPOINT_EVERY  = 50    # save every N embed batches (~6 400 passages at batch=128)
EMPTY_CACHE_EVERY = 200   # torch.cuda.empty_cache() every N batches
UPSERT_BATCH      = 500   # points per Qdrant upsert call
TOTAL_PASSAGES    = 25_247_890


# -- Checkpoint helpers -------------------------------------------------------
def _save_ckpt(path, n):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps({"passages_done": n}))
    tmp.replace(path)

def _load_ckpt(path):
    return json.loads(path.read_text())["passages_done"] if path.exists() else 0


# -- Passage batcher with resume skip ----------------------------------------
def _batches(skip, batch_size):
    gen = iter_passages(WIKI_DIR)
    if skip:
        print(f"  Skipping {skip:,} already-indexed passages ...")
        collections.deque(itertools.islice(gen, skip), maxlen=0)
    buf_p, buf_t = [], []
    for pid, _page, _idx, text in gen:
        buf_p.append(pid)
        buf_t.append(text)
        if len(buf_p) == batch_size:
            yield buf_p, buf_t
            buf_p, buf_t = [], []
    if buf_p:
        yield buf_p, buf_t


# -- Upsert worker (runs in background thread) --------------------------------
def _upsert(points):
    client.upsert(collection_name=COLLECTION, points=points, wait=True)


# -- Symlink guard ------------------------------------------------------------
assert Path("rag/indices/qdrant").is_symlink(), "rag/indices/qdrant is not a symlink -- check storage"

# -- Disk space probe (defends against the v5-small disk-full crash mode) --
import shutil
qdrant_target = Path("rag/indices/qdrant").resolve()
free_gb       = shutil.disk_usage(qdrant_target).free / 1e9
MIN_FREE_GB   = 100   # 25M x ~2.6 KB ~= 65 GB + HNSW graph overhead
print(f"Free space on Qdrant volume : {free_gb:.1f} GB  (need >= {MIN_FREE_GB} GB)")
if free_gb < MIN_FREE_GB:
    raise RuntimeError(
        f"Insufficient disk space: {free_gb:.1f} GB free on {qdrant_target}, "
        f"need >= {MIN_FREE_GB} GB for 4C + HNSW build"
    )

# -- Resume -------------------------------------------------------------------
start_at = _load_ckpt(CHECKPOINT_PATH)
print(f"Starting 4C  |  passages_done={start_at:,}  ({start_at / TOTAL_PASSAGES * 100:.2f}%)")
print(f"Collection points before run : {client.count(COLLECTION).count:,}")

# -- Main loop ----------------------------------------------------------------
t0             = time.time()
passages_done  = start_at
batch_idx      = 0
pending_future = None

with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    pbar = tqdm(
        total     = TOTAL_PASSAGES,
        initial   = start_at,
        desc      = "4C",
        unit      = " p",
        smoothing = 0.05,
    )

    for pids, texts in _batches(start_at, EMBED_BATCH):

        # -- Embed on GPU (raw text -- no prefix) --
        # Background thread upserts batch N-1 concurrently during this call.
        vecs_np = harrier_encode(texts)   # shape (EMBED_BATCH, 640), FP32

        # -- Build Qdrant points --
        base_id = passages_done
        points  = [
            PointStruct(
                id      = base_id + j,
                vector  = vecs_np[j].tolist(),
                payload = {"passage_id": pids[j], "text": texts[j]},
            )
            for j in range(len(pids))
        ]

        # -- Gate: wait for upsert N-1, fire upsert N in background --
        if pending_future is not None:
            pending_future.result()
        pending_future = executor.submit(_upsert, points)

        passages_done += len(pids)
        batch_idx     += 1
        pbar.update(len(pids))

        elapsed = time.time() - t0
        p_per_s = (passages_done - start_at) / elapsed if elapsed > 0 else 0
        eta_h   = (TOTAL_PASSAGES - passages_done) / p_per_s / 3600 if p_per_s > 0 else 0
        pbar.set_postfix({"p/s": f"{p_per_s:.0f}", "ETA_h": f"{eta_h:.1f}"})

        # -- Checkpoint --
        if batch_idx % CHECKPOINT_EVERY == 0:
            pending_future.result()
            pending_future = None
            _save_ckpt(CHECKPOINT_PATH, passages_done)

        # -- VRAM cleanup --
        if batch_idx % EMPTY_CACHE_EVERY == 0:
            torch.cuda.empty_cache()

    # Flush final in-flight upsert
    if pending_future is not None:
        pending_future.result()

    pbar.close()

# -- Final checkpoint ---------------------------------------------------------
_save_ckpt(CHECKPOINT_PATH, passages_done)
elapsed_total = time.time() - t0
print(f"\n4C done  |  {passages_done:,} passages  |  {elapsed_total / 3600:.2f} h")
print(f"Throughput : {(passages_done - start_at) / elapsed_total:.0f} p/s (wall-clock)")
print(f"Collection count : {client.count(COLLECTION).count:,}")

# -- Trigger one-shot HNSW build ----------------------------------------------
print("\nLowering indexing_threshold -> 20,000 to trigger HNSW build ...")
client.update_collection(
    collection_name   = COLLECTION,
    optimizers_config = OptimizersConfigDiff(indexing_threshold=20_000),
)
info = client.get_collection(COLLECTION)
print(f"optimizer.indexing_threshold : {info.config.optimizer_config.indexing_threshold:,}")
print(f"collection.status            : {info.status}")


Free space on Qdrant volume : 472.5 GB  (need >= 100 GB)
Starting 4C  |  passages_done=0  (0.00%)
Collection points before run : 0


4C:   0%|          | 0/25247890 [00:00<?, ? p/s]

### Step 4D - HNSW build status poll

Polls the Qdrant collection status every 60 seconds until the HNSW index build triggered at the end of Step 4C completes. This is a standalone recovery cell -- safe to re-run after a kernel restart. The HNSW build continues in Qdrant even if the kernel is restarted.

Step 4C triggered the build by lowering `indexing_threshold` to 20 000 (below corpus size). The index is ready when `status` reports `green`. Expected build time for 25.2 M 640-dim vectors is 1--3 h after upload completes.


In [ ]:
# Cell 4D -- Poll HNSW build status until green
import time
from qdrant_client import QdrantClient

QDRANT_PATH = "rag/indices/qdrant"
COLLECTION  = "fever_wiki"
POLL_EVERY  = 60   # seconds between status checks

try:
    client
except NameError:
    client = QdrantClient(path=QDRANT_PATH)

print(f"Polling HNSW status every {POLL_EVERY}s -- safe to Ctrl+C, index keeps building.\n")
t0 = time.time()
while True:
    status    = client.get_collection(COLLECTION).status
    elapsed_h = (time.time() - t0) / 3600
    print(f"  [{elapsed_h:5.2f} h] status = {status}")
    if str(status).lower().endswith("green"):
        print(f"\nHNSW build complete in {elapsed_h:.2f} h.")
        break
    time.sleep(POLL_EVERY)
